In [5]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

In [6]:
PATH = "Weekly Pengeluaran Per Kapita 2024-2025.csv"  # atau pakai path lengkap
df = pd.read_csv(PATH)
df.columns = df.columns.str.strip()

GROUP_COL = "Kategori Makanan"
VALUE_COL = "Rata-Rata Pengeluaran Per Kapita"
DATE_COL = "Tanggal Minggu Awal"

# parse tanggal (format di file kamu: dd/mm/yyyy)
df[DATE_COL] = pd.to_datetime(df[DATE_COL], dayfirst=True, errors="coerce")
df = df.sort_values([GROUP_COL, DATE_COL]).reset_index(drop=True)

# ====== METHOD 1: IQR (per kategori) ======
k = 1.5  # 1.5 standar; naikkan (mis. 3.0) kalau mau lebih "longgar"
q1 = df.groupby(GROUP_COL)[VALUE_COL].transform(lambda s: s.quantile(0.25))
q3 = df.groupby(GROUP_COL)[VALUE_COL].transform(lambda s: s.quantile(0.75))
iqr = q3 - q1

df["iqr_low"] = q1 - k * iqr
df["iqr_high"] = q3 + k * iqr
df["outlier_iqr"] = (df[VALUE_COL] < df["iqr_low"]) | (df[VALUE_COL] > df["iqr_high"])

# ====== METHOD 2: Robust Z-score via MAD (per kategori) ======
# z_robust = 0.6745 * (x - median) / MAD ; threshold umum: 3.5
med = df.groupby(GROUP_COL)[VALUE_COL].transform("median")
mad = df.groupby(GROUP_COL)[VALUE_COL].transform(lambda s: np.median(np.abs(s - np.median(s))))

eps = 1e-9  # biar aman kalau MAD=0
df["z_robust"] = 0.6745 * (df[VALUE_COL] - med) / (mad + eps)
df["outlier_robust"] = df["z_robust"].abs() > 3.5

# ====== gabungan flag ======
df["outlier_any"] = df["outlier_iqr"] | df["outlier_robust"]

# ====== ringkasan & save ======
print("Outliers per kategori (IQR):")
print(df.groupby(GROUP_COL)["outlier_iqr"].sum().sort_values(ascending=False))

print("\nOutliers per kategori (Robust Z):")
print(df.groupby(GROUP_COL)["outlier_robust"].sum().sort_values(ascending=False))

outliers = df[df["outlier_any"]].copy()
outliers = outliers.sort_values([GROUP_COL, DATE_COL])

# simpan hasil lengkap + list outlier
df.to_csv("weekly_with_outlier_flags.csv", index=False)
outliers.to_csv("weekly_outliers_only.csv", index=False)

print(f"\nSaved: weekly_with_outlier_flags.csv ({len(df)} rows)")
print(f"Saved: weekly_outliers_only.csv ({len(outliers)} rows)")

Outliers per kategori (IQR):
Kategori Makanan
Bumbu-bumbuan/Spices                         3
Daging/Meat                                  3
Jumlah makanan/Total food                    3
Konsumsi lainnya/Miscellaneous food items    3
Padi-padian/Cereals                          3
Telur dan susu/Eggs and milk                 3
Name: outlier_iqr, dtype: int64

Outliers per kategori (Robust Z):
Kategori Makanan
Padi-padian/Cereals                          3
Telur dan susu/Eggs and milk                 3
Konsumsi lainnya/Miscellaneous food items    2
Bumbu-bumbuan/Spices                         2
Jumlah makanan/Total food                    2
Daging/Meat                                  1
Name: outlier_robust, dtype: int64

Saved: weekly_with_outlier_flags.csv (636 rows)
Saved: weekly_outliers_only.csv (18 rows)


In [8]:
import pandas as pd

df = pd.read_csv("weekly_with_outlier_flags.csv")
df.columns = df.columns.str.strip()

# mapping: kolom tanggal minggu awal -> week_start_date (yang dimau script lu)
df = df.rename(columns={"Tanggal Minggu Awal": "week_start_date"})

df["week_start_date"] = pd.to_datetime(df["week_start_date"], dayfirst=True, errors="coerce")

print(df.columns)
print(df["week_start_date"].isna().sum(), "tanggal gagal diparse")

Index(['Kategori Makanan', 'Tahun', 'week_start_date', 'Tanggal Minggu Akhir',
       'Rata-Rata Pengeluaran Per Kapita', 'iqr_low', 'iqr_high',
       'outlier_iqr', 'z_robust', 'outlier_robust', 'outlier_any'],
      dtype='object')
0 tanggal gagal diparse


In [9]:
df = pd.read_csv(
    "weekly_with_outlier_flags.csv",
    dtype={"Tanggal Minggu Awal": "string", "Tanggal Minggu Akhir": "string"}
)
df.columns = df.columns.str.strip()

START_COL = "Tanggal Minggu Awal"
END_COL   = "Tanggal Minggu Akhir"

def parse_date_series(s, dayfirst=True):
    raw = s.astype("string").str.strip()

    # pass 1: parse umum
    dt = pd.to_datetime(raw, dayfirst=dayfirst, errors="coerce")

    # pass 2: ISO yyyy-mm-dd
    dt_iso = pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce")
    dt = dt.fillna(dt_iso)

    # pass 3: Excel serial number
    num = pd.to_numeric(raw, errors="coerce")
    mask_excel = dt.isna() & num.between(20000, 60000)
    dt.loc[mask_excel] = pd.to_datetime(
        num.loc[mask_excel], unit="D", origin="1899-12-30", errors="coerce"
    )

    return dt, raw

# --- parse start & end (dayfirst default Indonesia) ---
start_dt, start_raw = parse_date_series(df[START_COL], dayfirst=True)
end_dt, end_raw     = parse_date_series(df[END_COL],   dayfirst=True)

# fallback: kalau start NaT tapi end kebaca -> start = end - 6 hari
mask = start_dt.isna() & end_dt.notna()
start_dt.loc[mask] = end_dt.loc[mask] - pd.Timedelta(days=6)

# sanity check: delta minggu normalnya 6 atau 7
delta = (end_dt - start_dt).dt.days

# kalau delta aneh, coba ulang parse baris itu dengan dayfirst=False (format US)
bad_delta = end_dt.notna() & start_dt.notna() & ~delta.isin([6, 7])
if bad_delta.any():
    start_dt_us, _ = parse_date_series(start_raw[bad_delta], dayfirst=False)
    end_dt_us, _   = parse_date_series(end_raw[bad_delta],   dayfirst=False)
    delta_us = (end_dt_us - start_dt_us).dt.days

    fix = delta_us.isin([6, 7])
    idx = bad_delta[bad_delta].index[fix]
    start_dt.loc[idx] = start_dt_us.loc[idx]
    end_dt.loc[idx]   = end_dt_us.loc[idx]

# simpan hasil parse final
df["Tanggal Minggu Awal Parsed"] = start_dt
df["Tanggal Minggu Akhir Parsed"] = end_dt

print("Start NaT:", df["Tanggal Minggu Awal Parsed"].isna().sum(), "/", len(df))
print("End NaT  :", df["Tanggal Minggu Akhir Parsed"].isna().sum(), "/", len(df))
print("\nDelta days counts:")
print(((df["Tanggal Minggu Akhir Parsed"] - df["Tanggal Minggu Awal Parsed"]).dt.days)
      .value_counts(dropna=False).head(10))

# contoh yang masih gagal total
bad = df[df["Tanggal Minggu Awal Parsed"].isna() & df["Tanggal Minggu Akhir Parsed"].isna()]
print("\nRows still unparsed (sample):")
print(bad[[START_COL, END_COL, "Tahun", "Kategori Makanan"]].head(20))

df.to_csv("weekly_with_parsed_dates.csv", index=False)
print("\nSaved: weekly_with_parsed_dates.csv")

Start NaT: 0 / 636
End NaT  : 0 / 636

Delta days counts:
6    636
Name: count, dtype: int64

Rows still unparsed (sample):
Empty DataFrame
Columns: [Tanggal Minggu Awal, Tanggal Minggu Akhir, Tahun, Kategori Makanan]
Index: []

Saved: weekly_with_parsed_dates.csv


In [16]:
import re
import pandas as pd
import numpy as np

INPUT_PATH = "weekly_with_parsed_dates.csv"
OUT_WEEKLY = "expenditure_weekly_clean.csv"        # row-level (jumlah baris sama kayak input)
OUT_DAILY  = "expenditure_daily_stepwise.csv"      # row-level × 7

def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^\w_]", "", regex=True)
    )
    return df

def parse_idr(x):
    if pd.isna(x):
        return pd.NA
    if isinstance(x, (int, float)):
        return float(x)

    s = str(x).strip().lower()
    s = s.replace("rp", "").replace("idr", "")
    s = re.sub(r"[^\d,\.]", "", s)

    if s.count(".") > 0 and s.count(",") == 0:
        s = s.replace(".", "")
    elif s.count(",") > 0 and s.count(".") == 0:
        s = s.replace(",", "")
    else:
        s = s.replace(".", "").replace(",", "")

    return float(s) if s else pd.NA

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# 1) Load + normalize
df = pd.read_csv(INPUT_PATH)
df = normalize_cols(df)

# 2) Pakai kolom parsed yang bener (setelah normalize)
week_start_col = find_col(df, ["tanggal_minggu_awal_parsed"])
week_end_col   = find_col(df, ["tanggal_minggu_akhir_parsed"])
if not week_start_col or not week_end_col:
    raise ValueError(f"Kolom parsed tidak ketemu. Kolom ada: {list(df.columns)}")

df["ws"] = pd.to_datetime(df[week_start_col].astype("string").str.strip(), errors="coerce")
df["we"] = pd.to_datetime(df[week_end_col].astype("string").str.strip(), errors="coerce")

df["date_ok"] = df["ws"].notna() & df["we"].notna()
df["delta_days"] = (df["we"] - df["ws"]).dt.days

# Paksa window weekly = 7 hari kalau date_ok tapi delta bukan 6 (tanpa drop baris)
mask_fix = df["date_ok"] & df["delta_days"].notna() & (df["delta_days"] != 6)
df.loc[mask_fix, "we"] = df.loc[mask_fix, "ws"] + pd.Timedelta(days=6)
df.loc[mask_fix, "delta_days"] = 6
df["we_fixed"] = False
df.loc[mask_fix, "we_fixed"] = True

# 3) Region default DKI kalau gak ada
region_col = find_col(df, ["region", "wilayah", "kota", "kabupaten", "provinsi"])
df["region"] = df[region_col].astype(str).str.strip() if region_col else "DKI"

# 4) Kategori (tetap dipertahankan)
cat_col = find_col(df, ["kategori_makanan", "kategori", "category"])
if not cat_col:
    df["kategori_makanan"] = "unknown"
else:
    df["kategori_makanan"] = df[cat_col].astype(str).str.strip()

# 5) Nilai pengeluaran
value_col = find_col(df, ["ratarata_pengeluaran_per_kapita", "pengeluaran", "expenditure", "value", "nilai"])
if not value_col:
    raise ValueError(f"Kolom nilai pengeluaran tidak ketemu. Kolom ada: {list(df.columns)}")

df["expenditure_value"] = df[value_col].apply(parse_idr)
df["value_ok"] = df["expenditure_value"].notna()

# 6) SIMPAN WEEKLY (ROW-LEVEL) — jumlah baris sama dengan input
df["row_id"] = np.arange(len(df))  # biar identitas baris aman walau ada duplikat
weekly_rowlevel = df.copy()
weekly_rowlevel.to_csv(OUT_WEEKLY, index=False)
print(f"✅ Saved weekly row-level (no aggregation): {OUT_WEEKLY} ({len(weekly_rowlevel)} rows)")

# 7) EXPAND ROW-LEVEL -> DAILY (setiap baris weekly jadi 7 baris daily)
n = len(weekly_rowlevel)
offsets = np.tile(np.arange(7), n)

daily = weekly_rowlevel.loc[weekly_rowlevel.index.repeat(7)].copy()
daily["date"] = weekly_rowlevel["ws"].repeat(7).to_numpy() + pd.to_timedelta(offsets, unit="D")

# Kalau ws NaT, date akan NaT (tetap tidak menghapus baris)
daily.to_csv(OUT_DAILY, index=False)
print(f"✅ Saved daily row-level: {OUT_DAILY} ({len(daily)} rows)")

✅ Saved weekly row-level (no aggregation): expenditure_weekly_clean.csv (636 rows)
✅ Saved daily row-level: expenditure_daily_stepwise.csv (4452 rows)
